# Verbatim — Phase 0 MVP notebook

**What this proves (spec `specs/2026-09-07-verbatim.md`, Phase 0):** the deterministic RAG core
works end to end on real public-domain manuals —

`load 2 manuals -> Docling parse -> layout-aware chunk (tables kept whole) -> gte-small embed ->
Supabase pgvector store -> retrieval filtered to ONE version_id (k=12, no reranker) ->
answer with a citation contract -> verify -> correct abstention on an out-of-version question.`

This is throwaway scratch work — not module structure, no RLS, no HNSW, no `match_chunks` RPC.
Those are Phase 1/2. Here one developer owns logic + output in one file.

### Corpus (public-domain instruments, downloaded to `data/manuals/`, gitignored)

| version_id key | file | instrument | source |
|---|---|---|---|
| `PSS` | `pss.pdf` | Perceived Stress Scale (Cohen) — scoring rule + demographic norm table | https://www.slu.edu/medicine/family-medicine/-pdf/perceived-stress-scale.pdf |
| `PHQ` | `phq9_gad7.pdf` | PHQ / GAD-7 Instruction Manual — 4 tables incl. PHQ-9 severity cutpoints | https://archive.thepcc.org/sites/default/files/resources/instructions.pdf |

Both are explicitly public domain (PHQ manual p.8; PSS reprinted with ASA permission, Cohen 1983).

### Checkpoint

Runs top to bottom; **one good cited answer** (PSS norm/scoring question) and
**one correct "not found in this version"** (PHQ-9 cutpoint asked against the PSS version).
If Docling mangles the norm tables or retrieval bleeds across versions, resolve before Phase 1.

In [1]:
# --- Cell 1 · Config & provenance -------------------------------------------------
# Reads .env.local (never committed). Needs:
#   LLM_BASE_URL / LLM_API_KEY / LLM_MODEL   -> Groq openai/gpt-oss-120b (OpenAI-compatible)
#   SUPABASE_DB_URL                          -> postgresql://... direct/pooler connection
import os
import uuid
import json
import time
import textwrap
from pathlib import Path

from dotenv import dotenv_values

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
ENV = dotenv_values(REPO / ".env.local")

LLM_BASE_URL = ENV["LLM_BASE_URL"].rstrip("/")
LLM_API_KEY = ENV["LLM_API_KEY"]
LLM_MODEL = ENV["LLM_MODEL"]
SUPABASE_DB_URL = ENV.get("SUPABASE_DB_URL")

assert SUPABASE_DB_URL, (
    "SUPABASE_DB_URL missing from .env.local — add the direct/pooler connection string "
    "from Supabase dashboard > Connect (postgresql://postgres.<ref>:<pw>@...:5432/postgres)"
)

# Stable synthetic version ids so re-runs are deterministic (uuid5 off a fixed namespace).
NS = uuid.UUID("00000000-0000-0000-0000-000000000000")
MANUALS = [
    {
        "key": "PSS",
        "version_id": str(uuid.uuid5(NS, "PSS")),
        "path": REPO / "data/manuals/pss.pdf",
        "title": "Perceived Stress Scale (Cohen)",
        "source": "https://www.slu.edu/medicine/family-medicine/-pdf/perceived-stress-scale.pdf",
    },
    {
        "key": "PHQ",
        "version_id": str(uuid.uuid5(NS, "PHQ")),
        "path": REPO / "data/manuals/phq9_gad7.pdf",
        "title": "PHQ / GAD-7 Instruction Manual",
        "source": "https://archive.thepcc.org/sites/default/files/resources/instructions.pdf",
    },
]
BY_KEY = {m["key"]: m for m in MANUALS}

for m in MANUALS:
    assert m["path"].exists(), f"missing {m['path']} — re-download from {m['source']}"
    print(f"{m['key']}  {m['version_id']}  {m['path'].name}  ({m['path'].stat().st_size:,} B)")

print(f"\nLLM: {LLM_MODEL} @ {LLM_BASE_URL}")

PSS  28256d4b-e909-57f3-a2a8-eeb0a93795f7  pss.pdf  (15,890 B)
PHQ  e1c8bfed-661e-5688-b428-0ceb36a18053  phq9_gad7.pdf  (142,718 B)

LLM: openai/gpt-oss-120b @ https://api.groq.com/openai/v1


In [2]:
# --- Cell 2 · Docling parse ----------------------------------------------------------
# PDF -> structured DoclingDocument (text + tables + layout + per-item page provenance).
from docling.document_converter import DocumentConverter

converter = DocumentConverter()
docs = {}
for m in MANUALS:
    t0 = time.time()
    result = converter.convert(str(m["path"]))
    docs[m["key"]] = result.document
    n_tables = len(result.document.tables)
    n_texts = len(result.document.texts)
    pages = result.document.num_pages()
    print(f"{m['key']}: {pages} pages, {n_texts} text items, {n_tables} tables  "
          f"({time.time() - t0:.1f}s)")

[INFO] 2026-09-08 09:47:07,923 [RapidOCR] base.py:23: Using engine_name: torch


[INFO] 2026-09-08 09:47:07,927 [RapidOCR] device_config.py:57: Using CPU device


[INFO] 2026-09-08 09:47:07,937 [RapidOCR] download_file.py:60: File exists and is valid: /Users/chiragshome/projects/verbatim/.venv/lib/python3.11/site-packages/rapidocr/models/PP-OCRv6_det_small.pth


[INFO] 2026-09-08 09:47:07,938 [RapidOCR] main.py:50: Using /Users/chiragshome/projects/verbatim/.venv/lib/python3.11/site-packages/rapidocr/models/PP-OCRv6_det_small.pth


[INFO] 2026-09-08 09:47:08,060 [RapidOCR] base.py:23: Using engine_name: torch


...

2026-09-08 09:48:17,081 MatchingPostProcessor WARNING  Orphan pdf_cell 265 recovered to row=10 by nearest-row fallback (col=3, y=962.0, dist=109.3)


2026-09-08 09:48:17,081 MatchingPostProcessor WARNING  Orphan pdf_cell 266 recovered to row=10 by nearest-row fallback (col=3, y=962.0, dist=109.3)


2026-09-08 09:48:17,081 MatchingPostProcessor WARNING  Orphan pdf_cell 267 recovered to row=10 by nearest-row fallback (col=3, y=962.0, dist=109.3)


2026-09-08 09:48:17,081 MatchingPostProcessor WARNING  Orphan pdf_cell 268 recovered to row=10 by nearest-row fallback (col=4, y=962.0, dist=109.3)


PHQ: 9 pages, 91 text items, 4 tables  (50.8s)


In [3]:
# --- Cell 2b · Eyeball the norm/cutpoint tables (the risky bit) ---------------------
# Phase 0 checkpoint explicitly calls out "parsing mangles the norm tables". Look before trusting.
import pandas as pd

for key in ("PSS", "PHQ"):
    print(f"\n================  {key}  ================")
    for i, tbl in enumerate(docs[key].tables):
        page = tbl.prov[0].page_no if tbl.prov else "?"
        print(f"\n--- table {i} (page {page}) ---")
        try:
            print(tbl.export_to_dataframe(doc=docs[key]).to_string(max_rows=20))
        except Exception as e:
            print("df export failed:", e)
            print(tbl.export_to_markdown(doc=docs[key]))

# Expected (verified in the parse smoke test):
#   PSS table 0 / p1  -> demographic norm table, Male 12.1 / Female 13.7 / 18-29 14.2 ... INTACT
#   PHQ table 3 / p7  -> PHQ-9 Score | Severity | Treatment Actions, 0-4 .. 20-27       INTACT
#   PHQ table 1 / p3  -> versions table (PHQ-9 = 0-27, GAD-7 = 0-21)                    INTACT
#   PSS table 1 / p2  -> the questionnaire response grid, misdetected as a table & cell-
#                        duplicated. It is the item list, not a scoring table; the clean
#                        text is also present as text items. Acceptable for v1.



================  PSS  ================

--- table 0 (page 1) ---
          Category     N  Mean S.D.
0           Gender                 
1             Male   926  12.1  5.9
2           Female  1406  13.7  6.6
3              Age                 
4            18-29   645  14.2  6.2
5            30-44   750  13.0  6.2
6            45-54   285  12.6  6.1
7            55-64   282  11.9  6.9
8       65 & older   296  12.0  6.3
9             Race                 
10           white  1924  12.8  6.2
11        Hispanic    98  14.0  6.9
12           black   176  14.7  7.2
13  other minority    50  14.1  5.0

--- table 1 (page 2) ---
                                                                                                                              0 = Never 1 = Almost Never 2 = Sometimes 3 = Fairly                                                                                                                                                                           Often Often 4 = Ver

In [4]:
# --- Cell 3 · Layout-aware chunking (tables kept whole) ----------------------------
# Docling HybridChunker: merges undersized pieces, never splits a table, carries page + heading
# provenance. Token budget sized to the gte-small context (512).
from docling.chunking import HybridChunker
from transformers import AutoTokenizer

EMB_MODEL_ID = "thenlper/gte-small"
hf_tok = AutoTokenizer.from_pretrained(EMB_MODEL_ID)
chunker = HybridChunker(tokenizer=hf_tok, max_tokens=512, merge_peers=True)

rows = []  # each: version_id, key, page, section, content
for m in MANUALS:
    for ch in chunker.chunk(docs[m["key"]]):
        pages = sorted({p.page_no for it in ch.meta.doc_items for p in it.prov})
        headings = " > ".join(ch.meta.headings or [])
        rows.append({
            "version_id": m["version_id"],
            "key": m["key"],
            "page": pages[0] if pages else None,
            "section": headings,
            "content": chunker.contextualize(chunk=ch),  # heading-prefixed text sent to the embedder
        })

chunks_df = pd.DataFrame(rows)
print(chunks_df.groupby("key").agg(n_chunks=("content", "size"),
                                   avg_chars=("content", lambda s: int(s.str.len().mean()))))
print("\nsample PSS chunk mentioning the norm table:")
mask = (chunks_df.key == "PSS") & chunks_df.content.str.contains("Mean", case=False)
print(textwrap.indent(chunks_df[mask].iloc[0].content[:900], "  ") if mask.any() else "  (none — check chunking)")

     n_chunks  avg_chars
key                     
PHQ        25       1169
PSS        16        956

sample PSS chunk mentioning the norm table:
  Sheldon Cohen
  Norm Table for the PSS 10 item inventory

  Gender, N = . Gender, Mean = . Gender, S.D. = . Male, N = 926. Male, Mean = 12.1. Male, S.D. = 5.9. Female, N = 1406. Female, Mean = 13.7. Female, S.D. = 6.6. Age, N = . Age, Mean = . Age, S.D. = . 18-29, N = 645. 18-29, Mean = 14.2. 18-29, S.D. = 6.2. 30-44, N = 750. 30-44, Mean = 13.0. 30-44, S.D. = 6.2. 45-54, N = 285. 45-54, Mean = 12.6. 45-54, S.D. = 6.1. 55-64, N = 282. 55-64, Mean = 11.9. 55-64, S.D. = 6.9. 65 & older, N = 296. 65 & older, Mean = 12.0. 65 & older, S.D. = 6.3. Race, N = . Race, Mean = . Race, S.D. = . white, N = 1924. white, Mean = 12.8. white, S.D. = 6.2. Hispanic, N = 98. Hispanic, Mean = 14.0. Hispanic, S.D. = 6.9. black, N = 176. black, Mean = 14.7. black, S.D. = 7.2. other minority, N = 50. other minority, Mean = 14.1. other minority, S.D. = 5.0


In [5]:
# --- Cell 4 · Embed with gte-small (384-dim, both sides of the system) -------------
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMB_MODEL_ID)  # thenlper/gte-small
vecs = embedder.encode(
    chunks_df.content.tolist(),
    normalize_embeddings=True,     # cosine == dot; matches pgvector <=> usage below
    show_progress_bar=True,
    batch_size=32,
)
chunks_df["embedding"] = list(vecs)
print("embeddings:", vecs.shape, "dtype", vecs.dtype)
assert vecs.shape[1] == 384

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

embeddings: (41, 384) dtype float16


In [6]:
# --- Cell 5 · Store in Supabase pgvector (disposable scratch table) ---------------
# DROP + CREATE every run so this stays throwaway. No HNSW (tiny corpus -> exact scan is fine
# and avoids index-build tuning); no RLS (service-role connection). Both are Phase 1/2 concerns.
import psycopg
from pgvector.psycopg import register_vector

DDL = (
    "create extension if not exists vector;"
    "drop table if exists mvp_chunks;"
    "create table mvp_chunks ("
    "  id bigserial primary key,"
    "  version_id uuid not null,"
    "  key text not null,"
    "  page int,"
    "  section text,"
    "  content text not null,"
    "  embedding vector(384) not null"
    ");"
    "create index on mvp_chunks (version_id);"
)

with psycopg.connect(SUPABASE_DB_URL, autocommit=True) as conn:
    conn.execute(DDL)
    register_vector(conn)
    with conn.cursor() as cur:
        cur.executemany(
            "insert into mvp_chunks (version_id, key, page, section, content, embedding) "
            "values (%s,%s,%s,%s,%s,%s)",
            [
                (r.version_id, r.key, (None if pd.isna(r.page) else int(r.page)),
                 r.section, r.content, r.embedding)
                for r in chunks_df.itertuples()
            ],
        )
    counts = conn.execute(
        "select key, count(*) from mvp_chunks group by key order by key"
    ).fetchall()
print("stored:", dict(counts))

stored: {'PHQ': 25, 'PSS': 16}


In [7]:
# --- Cell 6 · Retrieval filtered to ONE version_id (k=12, no reranker) ------------
# This is the isolation boundary in miniature: the WHERE version_id = %s clause here becomes
# an RLS policy + the match_chunks security-invoker RPC in Phase 1/2. Same query shape.
K = 12

def retrieve(question, version_id, k=K):
    qv = embedder.encode([question], normalize_embeddings=True)[0]
    sql = (
        "select id, key, page, section, content, 1 - (embedding <=> %s) as score "
        "from mvp_chunks where version_id = %s "
        "order by embedding <=> %s limit %s"
    )
    with psycopg.connect(SUPABASE_DB_URL) as conn:
        register_vector(conn)
        return conn.execute(sql, (qv, version_id, qv, k)).fetchall()

_probe = retrieve("how is the scale scored", BY_KEY["PSS"]["version_id"])
print("top-5 for 'how is the scale scored' @ PSS version:")
for cid, key, page, section, content, score in _probe[:5]:
    print(f"  [{cid}] {key} p{page} score={score:.3f}  {content[:70]!r}")
assert {r[1] for r in _probe} == {"PSS"}, "version filter leaked!"

top-5 for 'how is the scale scored' @ PSS version:
  [5] PSS p2 score=0.821  'Perceived Stress Scale\n\n1. In the last month, how often have you been '
  [1] PSS p1 score=0.815  'Sheldon Cohen\nThe Perceived Stress Scale (PSS) is the most widely used'
  [14] PSS p2 score=0.815  'Perceived Stress Scale\n\n ...................................... 9. In '
  [15] PSS p2 score=0.813  'Perceived Stress Scale\n them?............................, Often = 0. '
  [13] PSS p2 score=0.813  'Perceived Stress Scale\n\n life?........................................'


In [8]:
# --- Cell 7 · Answer + verify: citation contract, JSON-parse-retry-abstain --------
# Mirrors supabase/functions/_shared/{llm,prompt}.ts intent. Notebook uses a subset of the
# spec's AnswerResult: {answer, citations:[{chunkId,page,quote}], abstained}. temperature 0.
import httpx

ABSTAIN_MSG = "Not found in this version."

ANSWER_SYS = (
    "You answer strictly and only from the provided CONTEXT chunks, which all come from a single\n"
    "version of a psychological test manual. Every factual claim must be backed by a chunk.\n"
    "If the CONTEXT does not contain the answer, you MUST abstain - do not use outside knowledge.\n\n"
    "Reply with ONE JSON object, nothing else:\n"
    '{"answer": str, "citations": [{"chunkId": int, "page": int, "quote": str}], "abstained": bool}\n'
    "- quote: a short verbatim span copied from the cited chunk.\n"
    '- If abstaining: answer = "Not found in this version.", citations = [], abstained = true.'
)

VERIFY_SYS = (
    "You check whether an ANSWER is fully supported by the CONTEXT chunks. A claim is supported\n"
    "only if a chunk states it. Reply with ONE JSON object, nothing else:\n"
    '{"supported": bool, "unsupportedClaims": [str]}'
)

def _first_json(text):
    i = text.find("{")
    if i == -1:
        raise ValueError("no JSON object")
    depth = 0
    for j in range(i, len(text)):
        depth += (text[j] == "{") - (text[j] == "}")
        if depth == 0:
            return json.loads(text[i:j + 1])
    raise ValueError("unbalanced JSON object")

_last_call = [0.0]  # module-level throttle so bursts stay under the Groq free-tier RPM/TPM

def _chat(system, user, _min_gap=2.0, _max_tries=5):
    # space calls out
    wait = _min_gap - (time.time() - _last_call[0])
    if wait > 0:
        time.sleep(wait)
    for attempt in range(_max_tries):
        r = httpx.post(
            f"{LLM_BASE_URL}/chat/completions",
            headers={"Authorization": f"Bearer {LLM_API_KEY}"},
            json={"model": LLM_MODEL, "temperature": 0,
                  "messages": [{"role": "system", "content": system},
                               {"role": "user", "content": user}]},
            timeout=120,
        )
        if r.status_code == 429 and attempt < _max_tries - 1:
            delay = float(r.headers.get("retry-after", 0)) or min(2 ** attempt * 5, 60)
            print(f"  429 from Groq — backing off {delay:.0f}s (attempt {attempt + 1}/{_max_tries})")
            time.sleep(delay)
            continue
        r.raise_for_status()
        _last_call[0] = time.time()
        return r.json()["choices"][0]["message"]["content"]
    r.raise_for_status()  # exhausted retries

def _chat_json(system, user):
    raw = _chat(system, user)
    try:
        return _first_json(raw)
    except Exception:
        raw = _chat(system, user + "\n\nReturn VALID JSON only. No prose, no code fences.")
        try:
            return _first_json(raw)
        except Exception:
            return None  # caller treats as abstention

def _fmt_context(hits):
    return "\n\n".join(
        f"[chunkId={cid}] (page {page})\n{content}" for cid, key, page, section, content, score in hits
    )

def answer(question, version_id):
    hits = retrieve(question, version_id)
    retrieved = [(cid, page, round(score, 3)) for cid, k_, page, s_, c_, score in hits]
    parsed = _chat_json(ANSWER_SYS, f"QUESTION:\n{question}\n\nCONTEXT:\n{_fmt_context(hits)}")
    if not parsed or parsed.get("abstained") or not parsed.get("citations"):
        return {"answer": ABSTAIN_MSG, "citations": [], "abstained": True, "retrieved": retrieved}
    valid_ids = {cid for cid, *_ in hits}
    cites = [c for c in parsed["citations"] if c.get("chunkId") in valid_ids]
    if not cites:  # every citation hallucinated a chunk id -> abstain
        return {"answer": ABSTAIN_MSG, "citations": [], "abstained": True, "retrieved": retrieved}
    v = _chat_json(VERIFY_SYS, f"ANSWER:\n{parsed['answer']}\n\nCONTEXT:\n{_fmt_context(hits)}") or {}
    return {
        "answer": parsed["answer"],
        "citations": cites,
        "abstained": False,
        "verify": {"supported": bool(v.get("supported")),
                   "unsupportedClaims": v.get("unsupportedClaims", [])},
        "retrieved": retrieved,
    }

def show(label, result):
    print(f"### {label}")
    print("abstained:", result["abstained"])
    print("answer:", result["answer"])
    for c in result["citations"]:
        print(f"  - chunk {c['chunkId']} p{c['page']}: {c['quote']!r}")
    if result.get("verify"):
        print("verify:", result["verify"])
    print("retrieved (chunkId, page, score):", result["retrieved"][:6])
    print()


In [9]:
# --- Cell 8 · Demo query 1 — ANSWERABLE (expect a cited answer) -------------------
q1 = "Which PSS items are reverse-scored, and what is the mean PSS-10 score for women?"
r1 = answer(q1, BY_KEY["PSS"]["version_id"])
show("Q1 @ PSS version — expect cited answer (items 4,5,7,8; women mean 13.7)", r1)

# Pipeline behaviour is asserted; factual correctness (items 4,5,7,8 / mean 13.7 / page 1)
# is eyeballed against the printout and recorded in the checkpoint cell.
assert r1["abstained"] is False, "Q1 should be answerable from the PSS manual"
assert r1["citations"], "Q1 must carry >=1 citation to a retrieved chunk"
assert r1["verify"]["supported"], f"verify flagged unsupported claims: {r1['verify']['unsupportedClaims']}"


### Q1 @ PSS version — expect cited answer (items 4,5,7,8; women mean 13.7)
abstained: False
answer: The PSS items that are reverse‑scored are items 4, 5, 7, and 8. The mean PSS‑10 score for women is 13.7.
  - chunk 1 p1: 'reversing responses ... to the four positively stated items (items 4, 5, 7, & 8)'
  - chunk 2 p1: 'Female, N = 1406. Female, Mean = 13.7.'
verify: {'supported': True, 'unsupportedClaims': []}
retrieved (chunkId, page, score): [(2, 1, 0.89), (3, 2, 0.843), (1, 1, 0.824), (15, 2, 0.811), (14, 2, 0.809), (5, 2, 0.802)]



In [10]:
# --- Cell 9 · Demo query 2 — CROSS-VERSION (expect abstention) -------------------
# The PHQ-9 severity cutpoints live in the PHQ manual. Asked against the PSS version, retrieval
# is filtered to PSS chunks only -> the model has no supporting context -> "Not found".
q2 = "What PHQ-9 total score indicates 'moderately severe' depression?"
r2 = answer(q2, BY_KEY["PSS"]["version_id"])
show("Q2 @ PSS version — expect abstention (PHQ-9 cutpoints are not in this version)", r2)

assert r2["abstained"] is True, "Q2 must abstain — PHQ-9 cutpoints are not in the PSS version"
assert r2["citations"] == []

  429 from Groq — backing off 11s (attempt 1/5)


### Q2 @ PSS version — expect abstention (PHQ-9 cutpoints are not in this version)
abstained: True
answer: Not found in this version.
retrieved (chunkId, page, score): [(1, 1, 0.824), (14, 2, 0.819), (3, 2, 0.801), (5, 2, 0.8), (15, 2, 0.799), (2, 1, 0.798)]



In [11]:
# --- Cell 10 · Symmetry check — same question IS answerable in its own version ----
# Guards against "it abstained because the pipeline is broken" vs "abstained for the right reason".
r2b = answer(q2, BY_KEY["PHQ"]["version_id"])
show("Q2 @ PHQ version — expect cited answer (15-19 / 'moderately severe')", r2b)

assert r2b["abstained"] is False, "the PHQ-9 cutpoint question must be answerable from the PHQ manual"
assert r2b["citations"], "Q2b must carry >=1 citation"
# Eyeball: the answer should say 15-19 and cite the PHQ-9 severity text (p.6) or Table 3 (p.7).


  429 from Groq — backing off 31s (attempt 1/5)


  429 from Groq — backing off 1s (attempt 2/5)


  429 from Groq — backing off 35s (attempt 1/5)


### Q2 @ PHQ version — expect cited answer (15-19 / 'moderately severe')
abstained: False
answer: A PHQ‑9 total score of 15 (i.e., scores in the 15‑19 range) indicates moderately severe depression.
  - chunk 32 p5: 'Scores of 5, 10, 15, and 20 represent cutpoints for mild, moderate, moderately severe and severe depression, respectively.'
  - chunk 35 p6: '15 - 19, Depression Severity = Moderately Severe.'
verify: {'supported': True, 'unsupportedClaims': []}
retrieved (chunkId, page, score): [(32, 5, 0.933), (33, 6, 0.915), (34, 6, 0.903), (20, 2, 0.901), (35, 6, 0.898), (27, 5, 0.897)]



In [12]:
# --- Cell 11 · Teardown — keep the scratch table disposable ----------------------
# Comment out the DROP if you want to poke at mvp_chunks after the run; Cell 5 recreates it anyway.
with psycopg.connect(SUPABASE_DB_URL, autocommit=True) as conn:
    conn.execute("drop table if exists mvp_chunks")
print("dropped mvp_chunks — nothing left in the Supabase project")

dropped mvp_chunks — nothing left in the Supabase project


## Checkpoint result — PASS (2026-09-08)

Clean top-to-bottom run (`jupyter nbconvert --execute`), scratch table dropped at the end.

- [x] **Docling table fidelity.** PSS demographic norm table (table 0, p1) intact — Male 926/12.1/5.9,
      Female 1406/13.7/6.6, all age & race rows. PHQ-9 severity table (table 3, p7) intact —
      0–4 None-minimal … 15–19 Moderately Severe … 20–27 Severe. PHQ versions table (p3) intact.
      *Not* mangled into prose. (The PSS p2 questionnaire grid is misdetected as a table & cell-
      duplicated — it's the item list, not a scoring table, and the clean text is also present.)
- [x] **Q1 — answerable, PSS version.** `abstained=False`. Answer: "items 4, 5, 7, and 8 …
      mean PSS-10 score for women is 13.7" — factually correct. 2 citations, both p1, verbatim
      quotes from real chunks. `verify.supported=True`.
- [x] **Q2 — cross-version, PHQ-9 question @ PSS version.** `abstained=True`, "Not found in this
      version." Retrieval returned only PSS chunk ids — **no cross-version leak**.
- [x] **Q2b — symmetry, same question @ PHQ version.** `abstained=False`. Answer: "15–19 range …
      moderately severe depression" — correct. Cited chunk 32 (p5, the cutpoint sentence) and
      chunk 35 (the serialized severity table). `verify.supported=True`. Confirms Q2's abstention
      was version isolation, not a broken pipeline.

### Deviations observed (mirrored into the spec's "Deviations from spec")

1. **Groq free-tier 429s.** A burst of 4 answer/verify calls trips Groq's rate limit. Added a
   throttle + `Retry-After`/exponential backoff to `_chat` (Cell 7). `_shared/llm.ts` and the
   Phase-2 eval runner need the same, and CI eval throughput is capped by it.
2. **Chunk page attribution off-by-one on merged chunks.** `HybridChunker` merged a p6 text item
   with the p7 severity table; the notebook attributes the chunk to `min(pages)` = 6, while the
   table's own `prov.page_no` is 7 (correct — matches the physical page). Phase-1 ingestion should
   attribute a chunk to the page of its dominant/first body item (the table), not the min across
   merged items — page citations are core to this product.
3. **`gte-small` returns float16.** `sentence-transformers` emitted `(41, 384) float16`; pgvector
   stored it fine. Phase-1 ingestion should cast to float32 for headroom / determinism.
